# Evals - Scheduling Advisor

1. The model **reads** the conversation and calls `sched` or `dont_sched`
2. The code **knows** the date
3. SQL **decides** what is free

The cells that call OpenAI run only if there is an `OPENAI_API_KEY` in the `.env` file

In [1]:
import json
import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

### add the project folder to sys.path, so the app package can be imported
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app").exists())
FOLDER = ROOT / "app" / "scheduling_advisor"
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_core.utils.function_calling import convert_to_openai_tool
from sklearn.metrics import accuracy_score, confusion_matrix

from app.scheduling_advisor.agents import get_llm
from app.scheduling_advisor.calendar_tools import ScheduleDB
from app.scheduling_advisor.scheduling_advisor import SchedulingAdvisor, list_dates

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY:", HAS_KEY)

OPENAI_API_KEY: False


In [2]:
def report(y_true, y_pred):
    passed = sum(p == t for p, t in zip(y_pred, y_true))
    total = len(y_true)

    print(f"passed: {passed}")
    print(f"failed: {total - passed}")
    print(f"total:  {total}")
    print(f"accuracy: {accuracy_score(y_true, y_pred):.2%}")


def show_confusion_matrix(y_true, y_pred, labels, title, cmap="Blues"):
    ### an answer that is not one of the labels ("No answer") is also counted
    labels = labels + sorted(set(y_pred) - set(labels))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)

    plt.figure(figsize=(6, 4))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap=cmap)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.title(title)
    plt.show()

    ### TP / FP / FN / TN of every label
    rows = []
    for i, label in enumerate(labels):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        rows.append({"label": label, "TP": TP, "FP": FP, "FN": FN, "TN": TN})
    return pd.DataFrame(rows)


def open_calendar():
    """A calendar for the evals where every slot is free, so only the understanding of the model is measured"""
    db = ScheduleDB(FOLDER / "tech_eval.db")
    db.create(closed_days=(), all_free=True)
    return db

## What the model gets

In [3]:
now = datetime(2024, 4, 3, 15, 12)
advisor = SchedulingAdvisor(get_llm() if HAS_KEY else None, open_calendar())

### the function that the model gets
sched, dont_sched = advisor.make_tools(now)
print(json.dumps(convert_to_openai_tool(sched), indent=2))

{
  "type": "function",
  "function": {
    "name": "sched",
    "description": "This is the right time for an interview: search the calendar. Leave out what the candidate did not mention.\nreason - what the candidate wants, one short sentence.\nagreed - true only if the candidate accepted or suggested one exact day and one exact hour.\ndays - one day that suits the candidate [YYYY-MM-DD], or a range [first day, last day].\nhours - one hour that suits the candidate [HH:MM], or a range [first hour, last hour].\nrefused - the days (YYYY-MM-DD) and the times (YYYY-MM-DD HH:MM) that the candidate turned down.",
    "parameters": {
      "properties": {
        "reason": {
          "type": "string"
        },
        "agreed": {
          "type": "boolean"
        },
        "days": {
          "default": [],
          "items": {
            "type": "string"
          },
          "type": "array"
        },
        "hours": {
          "default": [],
          "items": {
            "type"

In [4]:
### the dates that the model looks up, instead of calculating them
print(list_dates(now, days=12))

Wednesday 2024-04-03 (today, this week)
Thursday 2024-04-04 (tomorrow, this week)
Friday 2024-04-05 (this week)
Saturday 2024-04-06 (this week)
Sunday 2024-04-07 (next week)
Monday 2024-04-08 (next week)
Tuesday 2024-04-09 (next week)
Wednesday 2024-04-10 (next week)
Thursday 2024-04-11 (next week)
Friday 2024-04-12 (next week)
Saturday 2024-04-13 (next week)
Sunday 2024-04-14 (in 2 weeks)


## The code and the SQL, without the model

The arguments that the model should send, and what the advisor answers

In [5]:
wishes = {
    "no wish": {},
    "next Friday": {"days": ["2024-04-12"]},
    "next week, in the morning": {"days": ["2024-04-07", "2024-04-13"], "hours": ["09:00", "11:00"]},
    "next week on Tuesday or Thursday, after 4 PM": {"days": ["2024-04-09", "2024-04-11"], "hours": ["16:00", "17:00"], "refused": ["2024-04-10"]},
    "not on Thursday": {"refused": ["2024-04-04"]},
    "let's do Friday at 10 AM": {"agreed": True, "days": ["2024-04-05"], "hours": ["10:00"]},
    "let's do Friday at 10:30": {"agreed": True, "days": ["2024-04-05"], "hours": ["10:30"]},
    "yes to a day, without an hour (a mistake of the model)": {"agreed": True, "days": ["2024-04-05"]},
    "a time that already passed": {"agreed": True, "days": ["2024-04-01"], "hours": ["10:00"]},
    "not a date (a mistake of the model)": {"days": ["next Friday"], "hours": ["morning"]},
}

pd.set_option("display.max_colwidth", None)
pd.DataFrame([{"wish": wish, **advisor.find_slots(now, **inputs).model_dump()} for wish, inputs in wishes.items()])[["wish", "status", "reply_text"]]

,wish,status,reply_text
0,no wish,offer,"Thursday, April 4 at 9 AM, 10 AM or 11 AM"
1,next Friday,offer,"Friday, April 12 at 9 AM, 10 AM or 11 AM"
2,"next week, in the morning",offer,"Sunday, April 7 at 9 AM, 10 AM or 11 AM"
3,"next week on Tuesday or Thursday, after 4 PM",offer,"Tuesday, April 9 at 4 PM or 5 PM, or Thursday, April 11 at 4 PM"
4,not on Thursday,offer,"Friday, April 5 at 9 AM, 10 AM or 11 AM"
5,let's do Friday at 10 AM,agreed,"Friday, April 5 at 10 AM"
6,let's do Friday at 10:30,not available,"Friday, April 5 at 9 AM, 10 AM or 11 AM"
7,"yes to a day, without an hour (a mistake of the model)",offer,"Friday, April 5 at 9 AM, 10 AM or 11 AM"
8,a time that already passed,not available,"Wednesday, April 3 at 4 PM or 5 PM, or Thursday, April 4 at 10 AM"
9,not a date (a mistake of the model),no free slots,


In [6]:
### the same slot cannot be booked twice, and then it is not available any more
print(advisor.book("2024-04-05 10:00"), advisor.book("2024-04-05 10:00"))

advice = advisor.find_slots(now, days=["2024-04-05"], hours=["10:00"], agreed=True)
print(advice.status, "-", advice.reply_text)

True False
not available - Friday, April 5 at 9 AM, 11 AM or 12 PM


## The model: does it understand the candidate?

Cases that we wrote (`scheduling_cases.json`), each with the correct status and times. The calendar is open: all the hours are free

In [7]:
with open(FOLDER / "tests" / "scheduling_cases.json", "r", encoding="utf-8") as f:
    cases = json.load(f)

y_cases = [c["correct"] for c in cases]


def found(advice):
    """The status, and the times that the advisor found"""
    return f"{advice.status or advice.decision} {advice.reply_text}".strip()


pd.DataFrame([{"now": c["turns"][-1]["timestamp_utc"], "last message": c["turns"][-1]["text"], "correct": c["correct"]} for c in cases])

,now,last message,correct
0,2024-04-03T15:12,"The payments service of a bank, all in Python. I maintain it for a living.","offer Thursday, April 4 at 9 AM, 10 AM or 11 AM"
1,2024-04-03T15:12,"Gladly, I can do next Friday.","offer Friday, April 12 at 9 AM, 10 AM or 11 AM"
2,2024-04-03T15:12,This Friday at 1 PM would be better.,"agreed Friday, April 5 at 1 PM"
3,2024-04-03T15:12,The second option is great.,"agreed Thursday, April 4 at 2 PM"
4,2024-04-03T15:12,Sunday afternoon is better for me.,"offer Sunday, April 7 at 12 PM, 1 PM or 2 PM"
5,2024-04-03T15:12,"Tomorrow, but only after 2 PM.","offer Thursday, April 4 at 2 PM, 3 PM or 4 PM"
6,2024-04-03T15:12,April 16 at noon.,"agreed Tuesday, April 16 at 12 PM"
7,2024-04-03T15:12,What about the day after tomorrow at 4?,"agreed Friday, April 5 at 4 PM"
8,2024-04-03T15:12,I prefer Monday at 3 PM. How long does it take?,"agreed Monday, April 8 at 3 PM"
9,2024-04-03T15:12,"Thursday is impossible, I'm abroad until the evening.","offer Friday, April 5 at 9 AM, 10 AM or 11 AM"


In [8]:
if HAS_KEY:
    advisor.db = open_calendar()

    ### 3 runs on the same cases: a different answer means that something in the prompt is not defined well
    runs = [[advisor.propose_slots(c["turns"]) for c in cases] for run in range(3)]
    answers = pd.DataFrame([[found(advice) for advice in run] for run in runs])
    print("cases with different answers:", (answers.nunique() > 1).sum())

    y_found = list(answers.loc[0])
    report(y_cases, y_found)

    ### where did the advisor fail?
    for c, pred in zip(cases, y_found):
        if pred != c["correct"]:
            print(f'{c["turns"][-1]["text"]}\n    found: {pred}\n    correct: {c["correct"]}')

In [9]:
if HAS_KEY:
    ### the decision alone
    DECISIONS = ["Sched", "Don't Sched"]
    y_decision = [DECISIONS[0] if c["correct"].startswith(("offer", "agreed")) else DECISIONS[1] for c in cases]

    display(show_confusion_matrix(y_decision, [advice.decision for advice in runs[0]], DECISIONS, "Confusion Matrix - Scheduling Advisor"))